##  Environment Setup and Directory Creation
we start by importing the necessary data manipulation libraries and creating a dedicated output directory to store our cleaned CSV files.

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
output_dir = "Fianal_Data_Set"
os.makedirs(output_dir, exist_ok=True)

**Loading My Raw Datasets**

we read all the initial CSV files into Pandas DataFrames. we ensure that these files are located in the same directory as our script.

In [5]:
categories = pd.read_csv('./Data_Set/Categories.csv')
suppliers = pd.read_csv('./Data_Set/Suppliers.csv')
products = pd.read_csv('./Data_Set/Products.csv')
customers = pd.read_csv('./Data_Set/Customers.csv')
orders = pd.read_csv('./Data_Set/Orders.csv')
order_details = pd.read_csv('./Data_Set/Order_Details.csv')
payments = pd.read_csv('./Data_Set/Payments.csv')
shipments = pd.read_csv('./Data_Set/shipments (1).csv')

**Date Formatting Standardization**

Since our SQL Server is highly sensitive to date formats, we convert all date-related columns across our dataframes into a standardized YYYY-MM-DD string format to prevent import errors.

In [6]:
customers['JoinDate'] = pd.to_datetime(customers['JoinDate']).dt.strftime('%Y-%m-%d')
orders['OrderDate'] = pd.to_datetime(orders['OrderDate']).dt.strftime('%Y-%m-%d')
payments['PaymentDate'] = pd.to_datetime(payments['PaymentDate']).dt.strftime('%Y-%m-%d')

 **Orders Table Augmentation**
 
 project requires a minimum of 300 orders. If our current dataset falls short, we generate realistic mock orders and append them to my existing datafram

In [7]:
if len(orders) < 300:
    missing_orders_count = 300 - len(orders)
    last_order_id = orders['OrderID'].max()
    new_orders = pd.DataFrame({
        'OrderID': range(last_order_id + 1, last_order_id + 1 + missing_orders_count),
        'CustomerID': np.random.choice(customers['CustomerID'], missing_orders_count),
        'OrderDate': pd.date_range(start='2023-06-01', periods=missing_orders_count).strftime('%Y-%m-%d'),
        'OrderStatus': np.random.choice(['Completed', 'Processing', 'Shipped'], missing_orders_count)
    })
    orders = pd.concat([orders, new_orders], ignore_index=True)

**Order Details Table Augmentation**

 project requires a minimum of 800 order details. so we generate mock transaction items, linking them to my existing orders and products while calculating my derived LineTotal mathematically.

In [8]:
if len(order_details) < 800:
    missing_details = 800 - len(order_details)
    last_detail_id = order_details['OrderDetailID'].max()
    new_details = pd.DataFrame({
        'OrderDetailID': range(last_detail_id + 1, last_detail_id + 1 + missing_details),
        'OrderID': np.random.choice(orders['OrderID'], missing_details),
        'ProductID': np.random.choice(products['ProductID'], missing_details),
        'Quantity': np.random.randint(1, 5, missing_details),
        'UnitPrice': np.random.uniform(10, 500, missing_details).round(2),
        'Discount': 0,
    })
    new_details['LineTotal'] = new_details['Quantity'] * new_details['UnitPrice']
    order_details = pd.concat([order_details, new_details], ignore_index=True)

**Payments Table Augmentation**

project requires a minimum of 300 payment records. To satisfy my 1:1 business rule, so we
 identify orders without payments and generate corresponding payment transactions.

In [9]:
if len(payments) < 300:
    missing_payments = 300 - len(payments)
    last_pay_id = payments['PaymentID'].max()
    existing_orders_paid = payments['OrderID'].tolist()
    unpaid_orders = [oid for oid in orders['OrderID'] if oid not in existing_orders_paid][:missing_payments]
    
    new_payments = pd.DataFrame({
        'PaymentID': range(last_pay_id + 1, last_pay_id + 1 + len(unpaid_orders)),
        'OrderID': unpaid_orders,
        'AmountPaid': np.random.uniform(50, 1000, len(unpaid_orders)).round(2),
        'PaymentMethod': np.random.choice(['Credit Card', 'Cash', 'PayPal'], len(unpaid_orders)),
        'PaymentStatus': 'Completed',
        'PaymentDate': pd.date_range(start='2023-06-01', periods=len(unpaid_orders)).strftime('%Y-%m-%d')
    })
    payments = pd.concat([payments, new_payments], ignore_index=True)

**Shipments Table Transformation**

align shipments table with the business rules by taking exactly 300 rows, renaming the customer_id column to OrderID, and directly assigning valid Order IDs to enforce my 1:1 relationship.

In [ ]:
shipments = shipments.head(300).copy() 
shipments = shipments.rename(columns={'customer_id': 'OrderID'}) 
shipments['OrderID'] = orders['OrderID'].values[:300] 
shipments['shipment_date'] = pd.to_datetime(shipments['shipment_date']).dt.strftime('%Y-%m-%d')
shipments['delivery_date'] = pd.to_datetime(shipments['delivery_date']).dt.strftime('%Y-%m-%d')

**Exporting My Data for SQL Import**

 exporting all  augmented and cleaned DataFrames as new CSV files into my designated output directory without the index column.

In [11]:
categories.to_csv(f"{output_dir}/Categories.csv", index=False)
suppliers.to_csv(f"{output_dir}/Suppliers.csv", index=False)
products.to_csv(f"{output_dir}/Products.csv", index=False)
customers.to_csv(f"{output_dir}/Customers.csv", index=False)
orders.to_csv(f"{output_dir}/Orders.csv", index=False)
order_details.to_csv(f"{output_dir}/Order_Details.csv", index=False)
payments.to_csv(f"{output_dir}/Payments.csv", index=False)
shipments.to_csv(f"{output_dir}/Shipments.csv", index=False)